# Calculate APT inputs for 1 target

This rough notebook will go through some calculations to determing the APT inputs for a single target, specified at the start of the notebook. These values should be checked!

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import batman
from astropy.io import ascii, fits
import astropy.units as u
from astroquery.gaia import Gaia
from astroquery.simbad import Simbad
from astropy.time import Time
from astropy.coordinates import SkyCoord, EarthLocation
from astropy.modeling.models import BlackBody
from astropy.table import Column

import astrotools.generateExoplanetTable as GenTab
from astrotools.pandeia_miri_imaging_eclipse_calc import MIRIImaging_Observation_Eclipse
from astrotools.StarMotionCalculator import StarMotionCalculator
import astrotools.orbitparams as orb

Pandeia Engine version:  4.0
Pandeia RefData version:  4.0
Synphot Data:  /Users/hdiamondlowe/Programs/synphot_data/grp/redcat/trds/
Pandeia engine versions: None


### First, let's name our target and pull some data

In [2]:
hostname = "LTT 1445 A"
targname = "LTT 1445 A c"

In [3]:
systemspath = './systems/'
outputpath  = './outputs/'

In [4]:
try:
    system = ascii.read(f'{systemspath}/NASAExoArchive_ExoplanetSystem_{hostname.replace(" ", "")}.ecsv')
except(FileNotFound):
    system = GenTab.generateExoSystem(hostname=hostname, outputpath=systemspath)

In [5]:
system

hostname,pl_name,sy_dist,sy_disterr1,sy_disterr2,rastr,decstr,sy_vmag,sy_jmag,sy_kmag,st_mass,st_masserr1,st_masserr2,st_rad,st_raderr1,st_raderr2,st_teff,st_tefferr1,st_tefferr2,st_tefferr12,st_tefferr22,st_logg,st_lum,pl_tranmid,pl_tranmiderr1,pl_tranmiderr2,pl_rade,pl_radeerr1,pl_radeerr2,pl_bmasse,pl_bmasseerr1,pl_bmasseerr2,pl_orbper,pl_orbpererr1,pl_orbpererr2,pl_orbsmax,pl_orbsmaxerr1,pl_orbsmaxerr2,pl_orbincl,pl_imppar,pl_orbeccen,pl_orbeccenerr1,pl_orbeccenerr2,pl_eqt,pl_ratror,pl_ratrorerr1,pl_ratrorerr2,pl_ratdor,pl_ratdorerr1,pl_ratdorerr2,tran_flag,rv_flag
,,pc,,,sexagesimal,sexagesimal,,,,Solar mass,Solar mass,Solar mass,Solar Radius,Solar Radius,Solar Radius,K,K,K,K,K,log10(cm/s**2),log(Solar),days,days,days,Earth Radius,Earth Radius,Earth Radius,Earth Mass,,,days,days,days,,AU,AU,deg,,,,,K,,,,,,,,
object,object,float64,float64,float64,object,object,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int32,int32
LTT 1445 A,LTT 1445 A b,6.86929,0.003805,-0.003805,03h01m50.99s,-16d35m40.18s,10.59,7.294,6.496,0.26,0.01,-0.01,0.27,0.01,-0.01,3340.0,150.0,-150.0,150.0,-150.0,4.97,-2.046,2459152.2189,0.0005,-0.0005,1.18,0.06,-0.06,2.73,0.25,-0.23,5.35876,1e-05,-1e-05,0.022,0.003,-0.003,89.203,0.84,0.19,0.35,-0.14,555.0,0.0408,0.001,-0.001,29.963,3.128,-3.128,1,1
LTT 1445 A,LTT 1445 A c,6.86929,0.003805,-0.003805,03h01m50.99s,-16d35m40.18s,10.59,7.294,6.496,0.26,0.01,-0.01,0.27,0.01,-0.01,3340.0,150.0,-150.0,150.0,-150.0,4.95,-2.046,2458412.58159,0.00059,-0.00057,1.147,0.055,-0.054,1.54,0.2,-0.19,3.1239035,3.4e-06,-3.6e-06,0.02661,0.00047,-0.00049,87.43,0.937,0.223,--,--,508.0,0.0396,0.0018,-0.0017,21.56,0.78,-0.82,1,1


In [6]:
# We want to use mostly parameters from Pass, et al., 2023; 
# need to do this more by hand since not the default in NEA
if hostname == "LTT 1445 A":
    system_Pass = ascii.read(f"{systemspath}/LTT1445A_Pass.csv")

    for i in range(len(system_Pass)):
        system_Pass['pl_ratdor'][i] = ((system_Pass['pl_orbsmax'][i]*u.AU)/(system_Pass['st_rad'][i]*u.R_sun)).decompose().value
        system_Pass['st_teff'][i]   = 3340
        system_Pass['st_tefferr1'][i] = 150
        system_Pass['st_tefferr2'][i] = -150
        system_Pass['st_logg'][i] = 4.99
    system_Pass.add_column(Column(name='tran_flag', data=[1]*len(system_Pass), dtype=int))
    system_Pass.add_column(Column(name='rv_flag', data=[1]*len(system_Pass), dtype=int))

system_Pass

pl_name,hostname,pl_orbper,pl_orbpererr1,pl_orbpererr2,pl_orbperlim,pl_orbsmax,pl_orbsmaxerr1,pl_orbsmaxerr2,pl_orbsmaxlim,pl_rade,pl_radeerr1,pl_radeerr2,pl_radelim,pl_masse,pl_masseerr1,pl_masseerr2,pl_masselim,pl_bmasse,pl_bmasseerr1,pl_bmasseerr2,pl_bmasselim,pl_dens,pl_denserr1,pl_denserr2,pl_denslim,pl_orbeccen,pl_orbeccenerr1,pl_orbeccenerr2,pl_orbeccenlim,pl_insol,pl_insolerr1,pl_insolerr2,pl_insollim,pl_eqt,pl_eqterr1,pl_eqterr2,pl_eqtlim,pl_orbincl,pl_orbinclerr1,pl_orbinclerr2,pl_orbincllim,pl_tranmid,pl_tranmiderr1,pl_tranmiderr2,pl_tranmidlim,pl_imppar,pl_impparerr1,pl_impparerr2,pl_impparlim,pl_ratdor,pl_ratdorerr1,pl_ratdorerr2,pl_ratdorlim,pl_ratror,pl_ratrorerr1,pl_ratrorerr2,pl_ratrorlim,st_refname,st_teff,st_tefferr1,st_tefferr2,st_tefflim,st_rad,st_raderr1,st_raderr2,st_radlim,st_mass,st_masserr1,st_masserr2,st_masslim,st_met,st_meterr1,st_meterr2,st_metlim,st_logg,st_loggerr1,st_loggerr2,st_logglim,rastr,ra,decstr,dec,sy_dist,sy_disterr1,sy_disterr2,sy_vmag,sy_vmagerr1,sy_vmagerr2,sy_jmag,sy_jmagerr1,sy_jmagerr2,sy_kmag,sy_kmagerr1,sy_kmagerr2,sy_gaiamag,sy_gaiamagerr1,sy_gaiamagerr2,tran_flag,rv_flag
str12,str10,float64,float64,float64,int64,float64,float64,float64,int64,float64,float64,float64,int64,float64,float64,float64,int64,float64,float64,float64,int64,float64,float64,float64,int64,int64,int64,int64,int64,float64,float64,float64,int64,int64,int64,int64,int64,float64,float64,float64,int64,float64,float64,float64,int64,float64,float64,float64,int64,int64,int64,int64,int64,float64,float64,float64,int64,str126,int64,int64,int64,int64,float64,float64,float64,int64,float64,float64,float64,int64,int64,int64,int64,int64,int64,int64,int64,int64,str12,float64,str13,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64,int64
LTT 1445 A b,LTT 1445 A,5.3587635,4.4e-06,-4.5e-06,0,0.0381,0.00067,-0.0007,0,1.34,0.11,-0.06,0,2.73,0.25,-0.23,0,2.73,0.25,-0.23,0,6.2,1.2,-1.3,0,--,--,--,--,5.7,1.3,-1.1,0,431,23,-23,0,89.53,0.33,-0.4,0,2458412.7091,0.00047,-0.00046,0,0.25,0.18,-0.17,0,30,--,--,--,0.0454,0.0012,-0.0012,0,<a refstr=PASS_ET_AL__2023 href=https://ui.adsabs.harvard.edu/abs/2023AJ....166..171P/abstract target=ref>Pass et al. 2023</a>,3340,150,-150,--,0.27,0.02,-0.01,0,0.26,0.01,-0.01,0,--,--,--,--,4,--,--,--,03h01m50.99s,45.4624781,-16d35m40.18s,-16.5944956,6.86929,0.003805,-0.003805,10.59,0.046,-0.046,7.294,0.024,-0.024,6.496,0.021,-0.021,10.0469,0.001234,-0.001234,1,1
LTT 1445 A c,LTT 1445 A,3.1238994,3.1e-06,-3.3e-06,0,0.02659,0.00047,-0.00049,0,1.07,0.1,-0.07,0,1.37,0.19,-0.19,0,1.37,0.19,-0.19,0,5.9,1.8,-1.5,0,--,--,--,--,11.7,2.7,-2.3,0,516,28,-27,0,87.46,0.13,-0.21,0,2458412.58218,0.00078,-0.00074,0,0.937,0.012,-0.011,0,21,--,--,--,0.0362,0.0019,-0.0016,0,<a refstr=PASS_ET_AL__2023 href=https://ui.adsabs.harvard.edu/abs/2023AJ....166..171P/abstract target=ref>Pass et al. 2023</a>,3340,150,-150,--,0.27,0.02,-0.01,0,0.26,0.01,-0.01,0,--,--,--,--,4,--,--,--,03h01m50.99s,45.4624781,-16d35m40.18s,-16.5944956,6.86929,0.003805,-0.003805,10.59,0.046,-0.046,7.294,0.024,-0.024,6.496,0.021,-0.021,10.0469,0.001234,-0.001234,1,1


In [7]:
system = system_Pass
system

pl_name,hostname,pl_orbper,pl_orbpererr1,pl_orbpererr2,pl_orbperlim,pl_orbsmax,pl_orbsmaxerr1,pl_orbsmaxerr2,pl_orbsmaxlim,pl_rade,pl_radeerr1,pl_radeerr2,pl_radelim,pl_masse,pl_masseerr1,pl_masseerr2,pl_masselim,pl_bmasse,pl_bmasseerr1,pl_bmasseerr2,pl_bmasselim,pl_dens,pl_denserr1,pl_denserr2,pl_denslim,pl_orbeccen,pl_orbeccenerr1,pl_orbeccenerr2,pl_orbeccenlim,pl_insol,pl_insolerr1,pl_insolerr2,pl_insollim,pl_eqt,pl_eqterr1,pl_eqterr2,pl_eqtlim,pl_orbincl,pl_orbinclerr1,pl_orbinclerr2,pl_orbincllim,pl_tranmid,pl_tranmiderr1,pl_tranmiderr2,pl_tranmidlim,pl_imppar,pl_impparerr1,pl_impparerr2,pl_impparlim,pl_ratdor,pl_ratdorerr1,pl_ratdorerr2,pl_ratdorlim,pl_ratror,pl_ratrorerr1,pl_ratrorerr2,pl_ratrorlim,st_refname,st_teff,st_tefferr1,st_tefferr2,st_tefflim,st_rad,st_raderr1,st_raderr2,st_radlim,st_mass,st_masserr1,st_masserr2,st_masslim,st_met,st_meterr1,st_meterr2,st_metlim,st_logg,st_loggerr1,st_loggerr2,st_logglim,rastr,ra,decstr,dec,sy_dist,sy_disterr1,sy_disterr2,sy_vmag,sy_vmagerr1,sy_vmagerr2,sy_jmag,sy_jmagerr1,sy_jmagerr2,sy_kmag,sy_kmagerr1,sy_kmagerr2,sy_gaiamag,sy_gaiamagerr1,sy_gaiamagerr2,tran_flag,rv_flag
str12,str10,float64,float64,float64,int64,float64,float64,float64,int64,float64,float64,float64,int64,float64,float64,float64,int64,float64,float64,float64,int64,float64,float64,float64,int64,int64,int64,int64,int64,float64,float64,float64,int64,int64,int64,int64,int64,float64,float64,float64,int64,float64,float64,float64,int64,float64,float64,float64,int64,int64,int64,int64,int64,float64,float64,float64,int64,str126,int64,int64,int64,int64,float64,float64,float64,int64,float64,float64,float64,int64,int64,int64,int64,int64,int64,int64,int64,int64,str12,float64,str13,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64,int64
LTT 1445 A b,LTT 1445 A,5.3587635,4.4e-06,-4.5e-06,0,0.0381,0.00067,-0.0007,0,1.34,0.11,-0.06,0,2.73,0.25,-0.23,0,2.73,0.25,-0.23,0,6.2,1.2,-1.3,0,--,--,--,--,5.7,1.3,-1.1,0,431,23,-23,0,89.53,0.33,-0.4,0,2458412.7091,0.00047,-0.00046,0,0.25,0.18,-0.17,0,30,--,--,--,0.0454,0.0012,-0.0012,0,<a refstr=PASS_ET_AL__2023 href=https://ui.adsabs.harvard.edu/abs/2023AJ....166..171P/abstract target=ref>Pass et al. 2023</a>,3340,150,-150,--,0.27,0.02,-0.01,0,0.26,0.01,-0.01,0,--,--,--,--,4,--,--,--,03h01m50.99s,45.4624781,-16d35m40.18s,-16.5944956,6.86929,0.003805,-0.003805,10.59,0.046,-0.046,7.294,0.024,-0.024,6.496,0.021,-0.021,10.0469,0.001234,-0.001234,1,1
LTT 1445 A c,LTT 1445 A,3.1238994,3.1e-06,-3.3e-06,0,0.02659,0.00047,-0.00049,0,1.07,0.1,-0.07,0,1.37,0.19,-0.19,0,1.37,0.19,-0.19,0,5.9,1.8,-1.5,0,--,--,--,--,11.7,2.7,-2.3,0,516,28,-27,0,87.46,0.13,-0.21,0,2458412.58218,0.00078,-0.00074,0,0.937,0.012,-0.011,0,21,--,--,--,0.0362,0.0019,-0.0016,0,<a refstr=PASS_ET_AL__2023 href=https://ui.adsabs.harvard.edu/abs/2023AJ....166..171P/abstract target=ref>Pass et al. 2023</a>,3340,150,-150,--,0.27,0.02,-0.01,0,0.26,0.01,-0.01,0,--,--,--,--,4,--,--,--,03h01m50.99s,45.4624781,-16d35m40.18s,-16.5944956,6.86929,0.003805,-0.003805,10.59,0.046,-0.046,7.294,0.024,-0.024,6.496,0.021,-0.021,10.0469,0.001234,-0.001234,1,1


In [8]:
targind = np.argwhere(system['pl_name'] == targname)[0][0]
targ = system[targind]
targ

pl_name,hostname,pl_orbper,pl_orbpererr1,pl_orbpererr2,pl_orbperlim,pl_orbsmax,pl_orbsmaxerr1,pl_orbsmaxerr2,pl_orbsmaxlim,pl_rade,pl_radeerr1,pl_radeerr2,pl_radelim,pl_masse,pl_masseerr1,pl_masseerr2,pl_masselim,pl_bmasse,pl_bmasseerr1,pl_bmasseerr2,pl_bmasselim,pl_dens,pl_denserr1,pl_denserr2,pl_denslim,pl_orbeccen,pl_orbeccenerr1,pl_orbeccenerr2,pl_orbeccenlim,pl_insol,pl_insolerr1,pl_insolerr2,pl_insollim,pl_eqt,pl_eqterr1,pl_eqterr2,pl_eqtlim,pl_orbincl,pl_orbinclerr1,pl_orbinclerr2,pl_orbincllim,pl_tranmid,pl_tranmiderr1,pl_tranmiderr2,pl_tranmidlim,pl_imppar,pl_impparerr1,pl_impparerr2,pl_impparlim,pl_ratdor,pl_ratdorerr1,pl_ratdorerr2,pl_ratdorlim,pl_ratror,pl_ratrorerr1,pl_ratrorerr2,pl_ratrorlim,st_refname,st_teff,st_tefferr1,st_tefferr2,st_tefflim,st_rad,st_raderr1,st_raderr2,st_radlim,st_mass,st_masserr1,st_masserr2,st_masslim,st_met,st_meterr1,st_meterr2,st_metlim,st_logg,st_loggerr1,st_loggerr2,st_logglim,rastr,ra,decstr,dec,sy_dist,sy_disterr1,sy_disterr2,sy_vmag,sy_vmagerr1,sy_vmagerr2,sy_jmag,sy_jmagerr1,sy_jmagerr2,sy_kmag,sy_kmagerr1,sy_kmagerr2,sy_gaiamag,sy_gaiamagerr1,sy_gaiamagerr2,tran_flag,rv_flag
str12,str10,float64,float64,float64,int64,float64,float64,float64,int64,float64,float64,float64,int64,float64,float64,float64,int64,float64,float64,float64,int64,float64,float64,float64,int64,int64,int64,int64,int64,float64,float64,float64,int64,int64,int64,int64,int64,float64,float64,float64,int64,float64,float64,float64,int64,float64,float64,float64,int64,int64,int64,int64,int64,float64,float64,float64,int64,str126,int64,int64,int64,int64,float64,float64,float64,int64,float64,float64,float64,int64,int64,int64,int64,int64,int64,int64,int64,int64,str12,float64,str13,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64,int64
LTT 1445 A c,LTT 1445 A,3.1238994,3.1e-06,-3.3e-06,0,0.02659,0.00047,-0.00049,0,1.07,0.1,-0.07,0,1.37,0.19,-0.19,0,1.37,0.19,-0.19,0,5.9,1.8,-1.5,0,--,--,--,--,11.7,2.7,-2.3,0,516,28,-27,0,87.46,0.13,-0.21,0,2458412.58218,0.00078,-0.00074,0,0.937,0.012,-0.011,0,21,--,--,--,0.0362,0.0019,-0.0016,0,<a refstr=PASS_ET_AL__2023 href=https://ui.adsabs.harvard.edu/abs/2023AJ....166..171P/abstract target=ref>Pass et al. 2023</a>,3340,150,-150,--,0.27,0.02,-0.01,0,0.26,0.01,-0.01,0,--,--,--,--,4,--,--,--,03h01m50.99s,45.4624781,-16d35m40.18s,-16.5944956,6.86929,0.003805,-0.003805,10.59,0.046,-0.046,7.294,0.024,-0.024,6.496,0.021,-0.021,10.0469,0.001234,-0.001234,1,1


### Make an observation object for this target
In this step we can determine the correct subarray to use if our goal is XX% detector saturation. The outputs of this step will also tell us the calculated eclipse duration, which roughly tells us the total observation duration. We are not calculating the number of eclipses needed to reach a given scientific goal here.

In [9]:
obs = MIRIImaging_Observation_Eclipse(targ,
                                      filter='f1500w',
                                      subarray='full',
                                      frac_fullwell=0.75,
                                      nobs=1,
                                      find_best_subarray=True
)
timing = obs.get_target_timing()

  pl_name     hostname  pl_orbper pl_orbpererr1 pl_orbpererr2 pl_orbperlim pl_orbsmax pl_orbsmaxerr1 pl_orbsmaxerr2 pl_orbsmaxlim pl_rade pl_radeerr1 pl_radeerr2 pl_radelim pl_masse pl_masseerr1 pl_masseerr2 pl_masselim pl_bmasse pl_bmasseerr1 pl_bmasseerr2 pl_bmasselim pl_dens pl_denserr1 pl_denserr2 pl_denslim pl_orbeccen pl_orbeccenerr1 pl_orbeccenerr2 pl_orbeccenlim pl_insol pl_insolerr1 pl_insolerr2 pl_insollim pl_eqt pl_eqterr1 pl_eqterr2 pl_eqtlim pl_orbincl pl_orbinclerr1 pl_orbinclerr2 pl_orbincllim   pl_tranmid  pl_tranmiderr1 pl_tranmiderr2 pl_tranmidlim pl_imppar pl_impparerr1 pl_impparerr2 pl_impparlim pl_ratdor pl_ratdorerr1 pl_ratdorerr2 pl_ratdorlim pl_ratror pl_ratrorerr1 pl_ratrorerr2 pl_ratrorlim                                                           st_refname                                                           st_teff st_tefferr1 st_tefferr2 st_tefflim st_rad st_raderr1 st_raderr2 st_radlim st_mass st_masserr1 st_masserr2 st_masslim st_met st_meterr1 st_me

In [10]:
print(f"Best guess subarray for {targname} is {obs.subarray}")
print(f"Number of groups per integration is {obs.ngroups}")
print(f"Estimated eclipse duration (assuming e=0) is {obs.tdur.to(u.min):0.2f}")

Best guess subarray for LTT 1445 A c is sub64
Number of groups per integration is 21
Estimated eclipse duration (assuming e=0) is 31.12 min


### Get up-to-data Gaia coordinates and propoer motion

In [11]:
def get_gaia_dr3_data(star_name):
    print(star_name)
    star_aliases = Simbad.query_objectids(star_name)
    for alias in star_aliases:
        if 'gaia' in alias['ID'].lower() and 'dr3' in alias['ID'].lower():
            print(alias['ID'])
            gaia_source_id = alias['ID'].split(' ')[-1]
            print(gaia_source_id)
            
            query = f"SELECT source_id, ref_epoch, ra, dec, pm, pmra, pmdec, parallax, \
                    distance_gspphot, teff_gspphot, azero_gspphot, phot_g_mean_mag, radial_velocity \
                    FROM gaiadr3.gaia_source \
                    WHERE source_id = {gaia_source_id}"

            job     = Gaia.launch_job_async(query)
            results = job.get_results()
            print('    Reference epoch:')
            print('   ', results['ref_epoch'].value[0], results['ref_epoch'].unit)
            print('    RA    Dec:')
            coordinates = SkyCoord(results['ra'].value[0]*u.deg, results['dec'].value[0]*u.deg, frame='icrs')
            coords_string = coordinates.to_string('hmsdms')
            print(f'   {coords_string}')
            print('    Proper motion RA, Dec:')#, results['pmra'].unit, results['pmdec'].unit)
            print('   ', results['pmra'].value[0]*u.mas/u.yr, results['pmdec'].value[0]*u.mas/u.yr)
            print('    Parallax:')
            print('   ', (results['parallax'].value[0]*u.mas))

            results['input_star_name'] = star_name
            
    return results

In [12]:
gaia_coords = get_gaia_dr3_data(hostname)

LTT 1445 A
Gaia DR3 5153091836072107136
5153091836072107136
INFO: Query finished. [astroquery.utils.tap.core]
    Reference epoch:
    2016.0 yr
    RA    Dec:
   03h01m50.98188282s -16d35m40.31809984s
    Proper motion RA, Dec:
    -369.97184335074684 mas / yr -267.931311928784 mas / yr
    Parallax:
    145.69218853775283 mas


### Calcualte APT timing constraints

In [13]:
def calc_APT_phase(targ, tdur=None, obs_type='occ', Tfrac=1, Tcharge=1*u.hr, Tsettle=0.5*u.hr, Textra=0*u.hr, Tgamble=0*u.hr):
    '''
    targ     Dictionary formated like NASA Exopalnet Archive
    obs_type Can be 'tra' for a transit, 'occ' for an occultation; makes a difference in calculating phases
    Tfrac    The fraction of out of out-of-event/in-event time; e.g., for equal OOT to in-transit, tfrac=1
             tfrac=1 is a good conservative guess; very inadvisable to do tfrac < 1 for S/N reasons
    Tcharge  The amount of time in JWST overheads you get charged for time-sensitive observations
             Must have astropy unit
             Added to beginning of observations
             The observation can start any time in this 1-hour window; 1 hour comes from the APT so don't change
    Tsettle  The amount of time you think it will take the detector to settle down, different for each intrument
             Must have astropy unit
             Added to beginning of observations
             For MIRI Tsettle=30 mins
    Textra   Any extra time you want to add to the observation, e.g., to account for eclipse/transit time uncertainty
             Must have astropy unit
             Added equally to either side of the transit/occultation event
    Tgamble  Take this amount of time from the other times and put it after the event, essentially moving the observine window this much earlier
             Must have astropy unit
             This includes taking time from Tcharge and Tsettle such that you may end up with an event too close to the start
    !! Do not make any of the Times negative as this will break the code, duh.
    '''
    
    print(targ['pl_name'])
    if tdur == None: obs.Tdur_NEA(targ)

    # if transit duration < 1 hour, automatically add a little baseline to bring it up to 1 hr; 
    # COMMMENT OUT IF YOU DO NOT WANT THIS FEATURE
    #if tdur.to(u.min).value < 60:
    #    Tfrac = ((1*u.hr)/(tdur.to(u.hr))).decompose()
    #    print(f'   padding baseline for shorter transit/ Tfrac={Tfrac}')
    #per_thresh1, ecc_thresh1 = 3, 0.05
    #per_thresh2, ecc_thresh2 = 2, 0.05
    #if targ['pl_orbper'] > per_thresh1 and targ['pl_orbeccen'] >= ecc_thresh1:
        # worst case scenario
    #    Tfrac = 1.2
    #    print(f'   padding baseline for orbital period >{per_thresh1} days and ecc >= {ecc_thresh1}/ Tfrac={Tfrac}')
    #elif targ['pl_orbper'] > per_thresh2 and targ['pl_orbeccen'] >= ecc_thresh2:
    #    Tfrac = 1.0
    #    print(f'   padding baseline for orbital period >{per_thresh2} days and ecc >= {ecc_thresh2}/ Tfrac={Tfrac}')
    #if targ['pl_orbeccen'] >= 0.1:
    #    Tfrac = 1.5
    #    print(f'   padding baseline for large or uncertain ecc/ Tfrac={Tfrac}')
    if Textra.value > 0: print(f'   Adding {Textra} because of uncertain eclipse time')
    
    # T_14 in hours
    Tdur = tdur.to(u.hr)              # [hr]
    # Period in days
    P = targ['pl_orbper'] * u.day
    # T0; this is a time of mid-transit when phase=0 (does not have to be for a specific transit)
    T0 = targ['pl_tranmid']                # BJD-TDB
    # as a string, the RA and Dec of the host star, e.g., RA_Dec= '17:15:18.92 +04:57:50.1'
    RA_Dec = sys_coords = targ['rastr']+' '+targ['decstr']

    #############################################################################################################
    ######### magical code; please don't touch ##################################################################
    #############################################################################################################

    baseline = (Tfrac*Tdur).to(u.hr) + Textra # total time out of transit/eclipse

    # NB: Calculates assuming ecc=0. If you want to change that you can do some adjustments to the target parameters, 
    #     or update the calculation
    if   obs_type == 'tra': phase_midpoint = 0.0
    elif obs_type == 'occ': phase_midpoint = 0.5

    obs_start = Tdur/2 + baseline/2  # time before the event mid-point to start observing

    phase_max = phase_midpoint - (obs_start/P).decompose() - (Tsettle/P).decompose()
    phase_min = phase_max - (Tcharge/P).decompose()

    if Tgamble.value != 0: 
        print(f'   Gambling to take {Tgamble} before the event and stick it after (becuase observing windows seem to start on time)')
        phase_max += (Tgamble/P).decompose()
        phase_min += (Tgamble/P).decompose()

    # code shamelessly stolen from: https://gist.github.com/StuartLittlefair/4ab7bb8cf21862e250be8cb25f72bb7a
    # converts bjd_tdb to heliocentric times; need this for JWST APT
    def bary_to_helio(star, bjd, obs_name):
        bary = Time(bjd, scale='tdb', format='jd')
        obs = EarthLocation.of_site(obs_name)
        #star = SkyCoord(coords, unit=(u.hour, u.deg))
        ltt = bary.light_travel_time(star, 'barycentric', location=obs) 
        guess = bary - ltt
        delta = (guess + guess.light_travel_time(star, 'barycentric', obs)).jd  - bary.jd
        guess -= delta * u.d

        ltt = guess.light_travel_time(star, 'heliocentric', obs)
        return guess.utc + ltt

    phase0 = Time(T0, format='jd', scale='tdb')
    star = SkyCoord(RA_Dec, unit=(u.hourangle, u.deg))
    random_earth_place_name = 'lco' # for nostalgia

    phase0_hjd = bary_to_helio(star, phase0, random_earth_place_name)

    print('   ****** PHASE *******')
    print('   Phase range {} to {}'.format(phase_min, phase_max))
    print('   Period {}'.format(P))
    print('   Zero phase (HJD) {}'.format(phase0_hjd))
    print('')

    total_time = (Tcharge + Tsettle + Tdur + baseline).to(u.hr)

    print('   *** ESTIMATED TOTAL TIME ***')
    print('   NOTE: This does not go into the APT, it is just an estimate')
    print('   Event duration {}'.format(Tdur.to(u.min)))
    print('   Event duration {}'.format(Tdur.to(u.hr)))
    print('   TOTAL TIME PER OBSERVATION: {}'.format(total_time))
    print('')
    
    
    return total_time

In [14]:
def calc_integrations_per_exposure(ngroup, total_time, subarray):

    # only for MIRI Imaging 
    if   subarray.lower()=='full':      tframe = 2.77504 * u.s
    elif subarray.lower()=='brightsky': tframe = 0.86528 * u.s
    elif subarray.lower()=='sub256':    tframe = 0.29952 * u.s
    elif subarray.lower()=='sub128':    tframe = 0.11904 * u.s
    elif subarray.lower()=='sub64':     tframe = 0.08500 * u.s
    else: 
        print('ERROR: subarray not recognized')
        return
    
    tint    = tframe * ngroup                         # amount of time per integration
    treset  = 1*tframe                                # reset time between each integration
    cadence = tint + treset
    
    nintegrations = int(np.ceil((total_time/cadence).decompose()))
    print('    ***** FORM EDITOR ******')
    print(f'   subarray = {subarray}')
    print(f'   Ngroups {ngroup}')
    print(f'   Integrations/exposer {nintegrations}')
    
    return nintegrations

In [15]:
total_time = calc_APT_phase(targ, tdur=obs.tdur, Tfrac=1)
nintegrations = calc_integrations_per_exposure(obs.ngroups, total_time, subarray=obs.subarray)
#total_times += total_time*nobs

LTT 1445 A c
   ****** PHASE *******
   Phase range 0.47307487672160636 to 0.4864129076027118
   Period 3.1238994 d
   Zero phase (HJD) 2458412.5813583024

   *** ESTIMATED TOTAL TIME ***
   NOTE: This does not go into the APT, it is just an estimate
   Event duration 31.120382094191836 min
   Event duration 0.5186730349031973 h
   TOTAL TIME PER OBSERVATION: 2.5373460698063948 h

    ***** FORM EDITOR ******
   subarray = sub64
   Ngroups 21
   Integrations/exposer 4885


### Avoid sibling planet transits and occultations

In [16]:
start_time = Time("2024-12-15T00:00:00")
end_time   = Time("2027-02-15T00:00:00")

In [17]:
# using batman (Kreidberg+ 2015) to make eclipse parameters
def initialize_batman_model(targ):
    
    params = batman.TransitParams()       # object to store transit parameters
    params.t0  = 0.0                      # time of inferior conjunction
    params.per = targ['pl_orbper']        # orbital period (days)
    params.t_secondary = 0.5
    params.rp  = targ['pl_ratror']        # planet radius (in units of stellar radii)
    params.fp  = 0.0
    params.a   = targ['pl_ratdor']        # semi-major axis (in units of stellar radii)
    params.inc = targ['pl_orbincl']       # orbital inclination (in degrees)
    params.ecc = 0.                       # eccentricity
    params.w   = 90.                      # longitude of periastron (in degrees)
    params.limb_dark = "uniform"          # limb darkening model
    params.u = []                         # limb darkening coefficients [u1, u2, u3, u4]0

    return params

def get_batman_lightcurve(params, t, transittype):
    
    m = batman.TransitModel(params, t, transittype=transittype)
    flux = m.light_curve(params)

    return flux

def calc_FpFs(T_s, T_p, wavelength, Rp_Rs):
    
    ''' This function will take in the Temperature in Kelvin, 
    and the wavelength range that we are looking at,
    as well as the the radius of the star and the planet. '''
    
    bb_s = BlackBody(T_s, scale=1*u.erg/u.s/u.cm**2/u.AA/u.sr)
    bb_p = BlackBody(T_p, scale=1*u.erg/u.s/u.cm**2/u.AA/u.sr)
    
    Flux_ratio = bb_p(wavelength)/bb_s(wavelength) * (Rp_Rs)**2
        
    return Flux_ratio.decompose()

In [18]:
def make_APT_formated_time(astropy_time):
    year   = astropy_time.ymdhms.year
    month  = astropy_time.ymdhms.month
    day    = astropy_time.ymdhms.day
    hour   = astropy_time.ymdhms.hour
    minute = astropy_time.ymdhms.minute
    second = astropy_time.ymdhms.second
    
    if   month == 1:  monthname = 'JAN'
    elif month == 2:  monthname = 'FEB'
    elif month == 3:  monthname = 'MAR'
    elif month == 4:  monthname = 'APR'
    elif month == 5:  monthname = 'MAY'
    elif month == 6:  monthname = 'JUN'
    elif month == 7:  monthname = 'JUL'
    elif month == 8:  monthname = 'AUG'
    elif month == 9:  monthname = 'SEP'
    elif month == 10: monthname = 'OCT'
    elif month == 11: monthname = 'NOV'
    elif month == 12: monthname = 'DEC'
    else: monthname = 'Invalid Month'
        
    custom_time = f'{day:02}-{monthname}-{year}:{hour:02}:{minute:02}:{second:02.0f}'
    
    return custom_time

In [19]:
def get_bad_occultation(system, targind, tdur=None, start_time=Time.now(), end_time=Time.now()+365.25*u.day, plot=False, BETWEEN=False):
    '''
    system     Dictionary of a system from NASA Exoplanet Archive
    targind    The index of the planet that you want to target
    tdur       Eclipse (or transit) duation [astropy units]
    start_time The time you want to start checking for bad eclipses
               Default iw whatever time it is now
    end_time   The time you want to stop checking for bad eclipses
               Default iw whatever time it is now plus 1 year
    plot       True: plot out all the bad eclipses
               False: no plots
    BETWEEN    True: Formats times such that they can be put in the APT "BETWEEN" constraint
               False: Formats times in a more intuitive, human-readable way
    '''
    
    targ = system[targind]
    print(f"Making {targ['hostname']} system")
    print(f"Target planet is {targ['pl_name']}")
    print("Total number of planets = ", len(system))

    # get the transit numbers of events after the ephemeris that we are working with
    T0 = Time(targ['pl_tranmid'], format='jd', scale='utc')
    start_num = int(np.round(((start_time - T0)/targ['pl_orbper']).to(u.day).value))
    end_num   = int(np.round(((end_time - T0)/targ['pl_orbper']).to(u.day).value))
    event_num = np.arange(start_num, end_num+1)

    # get the eclipse numbers
    t0p5s = targ['pl_tranmid'] + (event_num+0.5)*targ['pl_orbper']

    print(f"    Looking for transit/occultation events between {start_time.to_string()} and {end_time.to_string()}")
    
    # get the transit/occultation events of the other planets in the system
    transit_events = []
    eclipse_events = []
    batman_params = []
    Fps = []
    for i, planet in enumerate(system):
        if i==targind: print("    ", i, planet['pl_name'], "--> target planet")
        else: print("    ", i, planet['pl_name'])
        if planet['tran_flag'] == 0: 
            print(f"    Careful, planet {other_planet['pl_name']} does not appear to transit")
            continue

        if planet['pl_ratror'] == None:
            print("Calculating Rp/Rs from separate Rp and Rs")
            Rp_Rs = ((planet['pl_rade']*u.R_earth)/(targ['st_rad']*u.R_sun)).decompose().value
        else: Rp_Rs = targ['pl_ratror']
        Fp = calc_FpFs(planet['st_teff']*u.K, planet['pl_eqt']*u.K, 15*u.um, planet['pl_ratror'])
        Fps.append(Fp)
        
        params = initialize_batman_model(planet)
        batman_params.append(params)      
        
        if i==targind: continue # this is the target, we are looking for events from other planets so move on
        
        T0 = Time(planet['pl_tranmid'], format='jd', scale='utc')
        start_num = int(np.round(((start_time - T0) / planet['pl_orbper']).to(u.day).value))
        end_num   = int(np.round(((end_time - T0) / planet['pl_orbper']).to(u.day).value))
        event_num = np.arange(start_num, end_num+1)

        transits = planet['pl_tranmid'] + event_num*planet['pl_orbper']
        eclipses = planet['pl_tranmid'] + (event_num+0.5)*planet['pl_orbper']
        
        transit_events.append(transits)
        eclipse_events.append(eclipses)

    transit_events = np.sort(np.hstack(transit_events))
    eclipse_events = np.sort(np.hstack(eclipse_events))
    
    other_events = np.array([transit_events, eclipse_events])

    # find the overlapping events with the target occultations
    if tdur == None: tdur = orb.Tdur_NEA(targ)
        
    n_bad_events = 0
    if BETWEEN: 
        good_time_start = start_time
        print("    BETWEEN values in APT")
    for t_occ in t0p5s:
        
        # how far from the center of the occultation to look
        #t_start = t_occ - tdur.to(u.day).value/2 - (1.5*u.hr).to(u.day).value  # take center of eclipse minus half a duration minus Tcharge+Tsettle=1.5 hours
        #t_end   = t_occ + tdur.to(u.day).value/2
        # !! Maybe think a bit harder about how to establish the "danger" window around the occultation
        t_start = t_occ - 3*tdur.to(u.day).value
        t_end   = t_occ + 3*tdur.to(u.day).value
        time = np.linspace(t_start, t_end, 1000)

        # get closest event to occulation
        # the logic here is that we just need to know if an occultation is bad, not where all of the other planets in the system are
        # so we only need to check for the next closest event, and if it falls in the potential observing window
        distance_to_t_occ = abs(t_occ - other_events)
        closest_dist = distance_to_t_occ.min()
        event_type, event_ind = np.unravel_index(distance_to_t_occ.argmin(), distance_to_t_occ.shape)
        closest_event = other_events[event_type,event_ind]

        if closest_event >= t_start and closest_event <= t_end:          

            for i, planet in enumerate(system):

                if i==targind:

                    batman_params[i].fp = Fps[i]
                    batman_params[i].t_secondary = t_occ
                    flux = get_batman_lightcurve(batman_params[i], time, transittype="secondary")
                    event_name = "eclipse"
                else:
                    if event_type==0:
                        batman_params[i].t0 = closest_event
                        flux = get_batman_lightcurve(batman_params[i], time, transittype="primary")
                        event_name = "transit"
                    elif event_type==1: 
                        batman_params[i].fp = Fps[i]
                        batman_params[i].t_secondary = closest_event
                        flux = get_batman_lightcurve(batman_params[i], time, transittype="secondary")
                        event_name = "eclipse"

                if plot: 
                    if i==targind: color='C1'
                    else: color='C0'
                    plt.plot(time, flux, color=color, lw=2, alpha=0.8, label=f"{planet['pl_name']} {event_name}")

            if plot:
                plt.legend()
                plt.xlabel('Time (days)')
                plt.ylabel('Relative Flux')
                plt.grid(alpha=0.3)
                plt.show()
            
            #print('    Do not observe between:', t_start, 'and', t_end)
            bad_time_start = Time(t_start, format='jd', scale='utc')
            bad_time_end   = Time(t_end, format='jd', scale='utc')            
            
            if BETWEEN:
                # this will instead print out the times where you can observe the eclipse
                print('        After date: ', make_APT_formated_time(good_time_start), 
                           '|  Before date: ', make_APT_formated_time(bad_time_start) )
                good_time_start = bad_time_end
            
            else:
                print('    Do not observe between:', bad_time_start.iso, 'and', bad_time_end.iso)
            n_bad_events += 1
            
    print(f"    Number of bad occultations = for {targ['pl_name']}: {n_bad_events} / {len(t0p5s)}; approx {int(np.round(n_bad_events/len(t0p5s)*100))} %")
    
#for target in sample: get_bad_occultation(target, BETWEEN=False)

In [20]:
get_bad_occultation(system, targind, tdur=obs.tdur, start_time=start_time, end_time=end_time, plot=False, BETWEEN=True)

Making LTT 1445 A system
Target planet is LTT 1445 A c
Total number of planets =  2
    Looking for transit/occultation events between 2024-12-15T00:00:00.000 and 2027-02-15T00:00:00.000
     0 LTT 1445 A b
     1 LTT 1445 A c --> target planet
    BETWEEN values in APT
        After date:  15-DEC-2024:00:00:00 |  Before date:  18-DEC-2024:18:53:05
        After date:  18-DEC-2024:21:59:48 |  Before date:  06-JAN-2025:12:43:34
        After date:  06-JAN-2025:15:50:18 |  Before date:  25-JAN-2025:06:34:04
        After date:  25-JAN-2025:09:40:47 |  Before date:  30-MAY-2026:17:25:14
        After date:  30-MAY-2026:20:31:58 |  Before date:  18-JUN-2026:11:15:44
        After date:  18-JUN-2026:14:22:27 |  Before date:  07-JUL-2026:05:06:13
        After date:  07-JUL-2026:08:12:57 |  Before date:  25-JUL-2026:22:56:43
        After date:  26-JUL-2026:02:03:26 |  Before date:  13-AUG-2026:16:47:12
        After date:  13-AUG-2026:19:53:56 |  Before date:  01-SEP-2026:10:37:42
        A

### Compute current location of target
This can be compared to outputs from the APT that show the target's motion; this is a way to check that the star is heading in the direction we think it should be. Something that looks off here could indicate that some coordinates or proper motions or somthing else is off.

In [21]:
# fill out the following for your target
####################
# THIS IS HOW MANY YEARS SINCE THE EPOCH YOU WANT TO MOVE THE STAR
t = 6 # elapsed time since reference epoch [yr]
####################

# get the gaia coordinates into a nice string
coordinates = SkyCoord(gaia_coords['ra'].value[0]*u.deg, gaia_coords['dec'].value[0]*u.deg, frame='icrs')
coords_string = coordinates.to_string('hmsdms')


starmotion = StarMotionCalculator(star_name      =gaia_coords['input_star_name'][0],
                                  coords_string  =coords_string,
                                  epoch          =gaia_coords['ref_epoch'].value[0],
                                  pmra           =gaia_coords['pmra'],
                                  pmdec          =gaia_coords['pmdec'],
                                  parallax       =gaia_coords['parallax'],
                                  radial_velocity=gaia_coords['radial_velocity'],
                                  t              =t
                                 )
c_new = starmotion.get_new_position()

Starting position of LTT 1445 A in 2016.0: 03h01m50.98188282s -16d35m40.31809984s
New position of LTT 1445 A in 2022.0: 03h01m50.82747469s -16d35m41.92555291s
